# Clasificación de textos por década — Parte 1: Machine Learning Clásico

**Competencia:** Aprendizaje de Máquina 2026-10  
**Tarea:** Predecir la **década** (primeros 3 dígitos del año) en que fue escrito un párrafo de texto histórico en español.  
**Clases:** Décadas del `150` al `188` (años 1500–1889), **39 clases** en total.  
**Restricción:** Únicamente modelos de `scikit-learn`, sin redes neuronales ni transformers.

## 1. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

try:
    from sklearn.experimental import enable_halving_search_cv  # noqa (sklearn < 1.0)
except ImportError:
    pass
from sklearn.model_selection import HalvingGridSearchCV

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = list(stopwords.words('spanish'))

SEED = 42
np.random.seed(SEED)

## 2. Carga de datos

In [ ]:
df_train = pd.read_csv('data/train.csv')
df_eval  = pd.read_csv('data/eval.csv')

print(f"Train: {df_train.shape}")
print(f"Eval:  {df_eval.shape}")
df_train.head(3)

## 3. Análisis Exploratorio (EDA)

In [ ]:
vc = df_train['decade'].value_counts().sort_index()

print(f"Décadas únicas : {df_train['decade'].nunique()}")
print(f"Rango          : {df_train['decade'].min()} – {df_train['decade'].max()}")
print(f"Ejemplos/clase : min={vc.min()}, max={vc.max()}, media={vc.mean():.0f}")

plt.figure(figsize=(14, 3))
plt.bar(vc.index, vc.values, color='steelblue')
plt.xlabel('Década'); plt.ylabel('Ejemplos')
plt.title('Distribución de ejemplos por década')
plt.tight_layout(); plt.show()

In [ ]:
# Longitud de los textos
lens = df_train['text'].str.len()
print(lens.describe())

plt.figure(figsize=(10, 3))
plt.hist(lens, bins=60, color='steelblue', edgecolor='white')
plt.xlabel('Longitud del texto (chars)'); plt.title('Distribución de longitud')
plt.tight_layout(); plt.show()

In [ ]:
# Muestra de textos por siglo
for decade in [150, 160, 170, 180, 188]:
    sample = df_train[df_train['decade'] == decade]['text'].iloc[0][:120]
    print(f"Década {decade}: {repr(sample)}\n")

## 4. Preprocesamiento del texto

Se aplica una limpieza **mínima** para preservar las señales ortográficas históricas que cambiaron entre los siglos XVI y XIX (e.g. uso de la 's' larga escrita como 'f', cambios en 'b/v', ortografía variante, etc.). Esas características son la clave para la clasificación temporal.

In [ ]:
def caracteres_mas_frecuentes(serie: pd.Series, top_n=10):
    # Unir todo el texto en un solo string
    texto = ''.join(serie.dropna().astype(str))
    texto = texto.lower()  # Convertir a minúsculas
    texto = re.sub(r'(\w+)([-¬>])\s+(\w+)', r'\1\3', texto) #Unir palabras con guiones
    texto = re.sub(r'[a-záéíóúüñ\s\d,.^:;\-\'*¿\?()¡!]', '', texto)  # Eliminar letras, espacios y dígitos
    
    # Contar frecuencia de cada carácter
    conteo = pd.Series(list(texto)).value_counts()
    
    # Retornar los más frecuentes
    return conteo.head(top_n)

In [ ]:
print(caracteres_mas_frecuentes(df_train['text'], top_n=20))

In [ ]:
def preprocess(text: str) -> str:
    """Limpieza mínima: colapsa saltos de línea y convierte a minúsculas."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = text.lower()
    text = re.sub(r'(\w+)([-¬>])\s+(\w+)', r'\1\3', text) #Unir palabras con guiones
    text = re.sub(r'[^a-záéíóúüñ\s\d,.^:;\-\'*¿\?]', '', text)
    return re.sub(r' +', ' ', text).strip()

df_train['clean_text'] = df_train['text'].apply(preprocess)
df_eval['clean_text']  = df_eval['text'].apply(preprocess)

random_idx = np.random.choice(df_train.index, size=5, replace=False)
print("Original:")
print(repr(df_train['text'].iloc[random_idx[0]][:200]))
print("\nLimpio:")
print(repr(df_train['clean_text'].iloc[random_idx[0]][:200]))

In [ ]:
X_train = df_train['clean_text'].values
y_train = df_train['decade'].values
X_eval  = df_eval['clean_text'].values

print(f"Train: {len(X_train)} ejemplos | Eval: {len(X_eval)} ejemplos")

## 5. Búsqueda de hiperparámetros con HalvingGridSearchCV

Se construye un **Pipeline único** compuesto por:

- **`TfidfVectorizer(analyzer='char', ngram_range=(3,5))`** — N-gramas de caracteres (cruza límites de palabra), captura patrones ortográficos y estilísticos temporales sin depender de palabras completas.
- **`SGDClassifier(loss='hinge')`** — equivalente a SVM lineal con descenso de gradiente estocástico; eficiente en vocabularios grandes.

Hiperparámetros a optimizar:

| Parámetro | Descripción | Valores |
|---|---|---|
| `tfidf__min_df` | Frecuencia mínima de documento para incluir un n-grama | 1, 2, 3, 5 |
| `clf__alpha` | Regularización L2 del SGDClassifier | 1e-5, 1e-4, 1e-3, 1e-2 |

`HalvingGridSearchCV` descarta rápidamente las configuraciones malas usando sucesivamente más datos, reduciendo el coste respecto a un grid search exhaustivo.

In [ ]:
# División hold-out estratificada (85% train / 15% val)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train
)
print(f"Train: {len(X_tr)} | Val: {len(X_val)}")

In [ ]:
# Pipeline: TF-IDF char n-gramas (3-5) + SGDClassifier(loss='hinge')
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=stop_words,
        analyzer='char',
        ngram_range=(3, 5),
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(max_iter=1000, solver='saga', class_weight='balanced', random_state=SEED))
])

param_grid = {
    'tfidf__min_df': [2, 3, 5],
    'clf__C': [1, 5, 10],
}

search = HalvingGridSearchCV(
    pipe,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=SEED,
)

print(f"Grid: {param_grid}")

In [ ]:
print("Ejecutando HalvingGridSearchCV...")
search.fit(X_tr, y_tr)

print(f"\nMejores hiperparámetros : {search.best_params_}")
print(f"CV accuracy (mejor)     : {search.best_score_:.4f}")

val_acc = accuracy_score(y_val, search.predict(X_val))
print(f"Val accuracy (hold-out) : {val_acc:.4f}")
print(f"Baseline aleatorio      : {1/39:.4f}")

In [ ]:
# Top configuraciones evaluadas por HalvingGridSearchCV
cv_df = pd.DataFrame(search.cv_results_)
top = (cv_df
       .nlargest(15, 'mean_test_score')
       [['param_clf__C', 'param_tfidf__min_df', 'mean_test_score', 'std_test_score']]
       .reset_index(drop=True))
print(top.to_string())

## 6. Entrenamiento del modelo final

Se extraen los mejores hiperparámetros encontrados por `HalvingGridSearchCV` y se reentrena el pipeline con **todos los datos de entrenamiento** (train + val).

In [ ]:
# Reentrenar con los mejores hiperparámetros sobre TODOS los datos
best_params = search.best_params_
print(f"Mejores parámetros: {best_params}")

best_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=stop_words,
        analyzer='char',
        ngram_range=(3, 5),
        sublinear_tf=True,
        min_df=best_params['tfidf__min_df'],
    )),
    ('clf', LogisticRegression(
        solver='saga',
        class_weight='balanced',
        max_iter=1000,
        C=best_params['clf__C'],
        random_state=SEED
    ))
])

print("Reentrenando sobre todos los datos de entrenamiento...")
best_pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, best_pipeline.predict(X_train))
print(f"Train accuracy (referencia) : {train_acc:.4f}")
print(f"Val   accuracy (hold-out)   : {val_acc:.4f}")

In [ ]:
# Reporte de clasificación sobre el hold-out (pipeline entrenado en X_tr)
val_pred = search.predict(X_val)
print(classification_report(y_val, val_pred))

## 7. Análisis de errores

In [ ]:
# ¿Cuánto se equivoca el modelo en número de décadas?
errors = np.abs(y_val.astype(int) - val_pred.astype(int))
print("Distribución de error (décadas de diferencia):")
for k in range(6):
    pct = (errors == k).mean() * 100
    print(f"  |error| = {k}:  {pct:.1f}%")
print(f"  |error| ≥ 6:  {(errors >= 6).mean()*100:.1f}%")

plt.figure(figsize=(10, 3))
plt.hist(errors, bins=range(0, 40), color='steelblue', edgecolor='white')
plt.xlabel('|Década real − Década predicha|')
plt.ylabel('Ejemplos'); plt.title('Distribución del error en décadas')
plt.tight_layout(); plt.show()

## 8. Guardar el modelo

In [ ]:
joblib.dump(best_pipeline, 'modelo_parte1.joblib')
print("Modelo guardado en: modelo_parte1.joblib")

# Verificación
loaded = joblib.load('modelo_parte1.joblib')
check = loaded.predict(X_train[:5])
print(f"Predicciones (primeros 5): {check}")
print(f"Etiquetas reales:          {y_train[:5]}")

## 9. Generación del archivo de respuesta (submission)

In [ ]:
eval_pred  = best_pipeline.predict(X_eval)

submission = pd.DataFrame({
    'id':     df_eval['id'],
    'answer': eval_pred
})

submission.to_csv('data/answers.csv', index=False)
print(f"data/answers.csv generado: {len(submission)} predicciones")
print(f"Rango de décadas predichas: {eval_pred.min()} – {eval_pred.max()}")
print(f"Décadas únicas predichas:   {len(np.unique(eval_pred))}")
print()
print(submission.head(10).to_string(index=False))


In [ ]:
# Distribución de predicciones vs entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

vc_train = pd.Series(y_train).value_counts().sort_index()
vc_pred  = pd.Series(eval_pred).value_counts().sort_index()

axes[0].bar(vc_train.index, vc_train.values, color='steelblue')
axes[0].set_title('Distribución train'); axes[0].set_xlabel('Década')

axes[1].bar(vc_pred.index, vc_pred.values, color='coral')
axes[1].set_title('Predicciones (eval)'); axes[1].set_xlabel('Década')

plt.tight_layout(); plt.show()

## Resumen

| Componente | Detalle |
|---|---|
| **Preprocesamiento** | Colapso de saltos de línea + minúsculas (preserva ortografía histórica) |
| **Features** | TF-IDF `char` (sin word-boundary): 3–5 gramas, `sublinear_tf=True`, `min_df` optimizado |
| **Modelo** | `SGDClassifier(loss='hinge')` — SVM lineal con SGD, eficiente para vocabularios grandes |
| **Búsqueda** | `HalvingGridSearchCV`: `min_df` ∈ {1,2,3,5} × `alpha` ∈ {1e-5,1e-4,1e-3,1e-2} |
| **Por qué `char`** | Cruza límites de palabra → captura patrones estilísticos temporales + robusto a OCR |
| **Validación** | Hold-out estratificado 15% — acc optimizada en 39 clases |
| **Modelo guardado** | `modelo_parte1.joblib` (joblib) |
| **Submission** | `data/answers.csv` (formato `id,answer`) |
